In [46]:
# House Price Regression - Multi-Model Predictions to Single Excel

import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor

In [47]:
# Load datasets
train_file = "data/train.csv"
test_file = "data/test.csv"

trainData = pd.read_csv(train_file)
testData = pd.read_csv(test_file)

In [48]:
# Extract target
y_train = trainData["SalePrice"].copy()
trainData = trainData.drop(["SalePrice"], axis=1)

In [49]:
# Combine for consistent preprocessing
combined = pd.concat([trainData, testData], keys=[0,1])

In [50]:
# Drop columns with >40% missing
threshold = int(0.6 * combined.shape[0])
combined = combined.dropna(thresh=threshold, axis=1)

In [51]:
# Impute missing values
numeric_cols = combined.select_dtypes(include=['int64','float64']).columns
for col in numeric_cols:
    combined[col].fillna(combined[col].mean(), inplace=True)

categorical_cols = combined.select_dtypes(exclude=['int64','float64']).columns
for col in categorical_cols:
    combined[col].fillna(combined[col].mode()[0], inplace=True)

C:\Users\knith\AppData\Local\Temp\ipykernel_25108\2646791440.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  combined[col].fillna(combined[col].mean(), inplace=True)
C:\Users\knith\AppData\Local\Temp\ipykernel_25108\2646791440.py:8: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained as

In [52]:
# One-hot encoding
combined = pd.get_dummies(combined, columns=categorical_cols)

In [53]:
# Separate train and test
x__train = combined.xs(0)
x_test = combined.xs(1)

# Preserve Id for final output
x_test_id = x_test["Id"].copy()

# Drop Id from features
x_train = x_train.drop(columns=["Id"], errors="ignore")
x_test_features = x_test.drop(columns=["Id"], errors="ignore")

# Fill any remaining missing values
x_train = x_train.fillna(0)
x_test_features = x_test_features.fillna(0)

In [54]:
# Log-transform target
y_train = np.log(y_train)

In [55]:
# Dictionary to store predictions
predictions = {}

# Helper function: train, predict, and store
def train_predict_store(model, model_name, x_train, y_train, x_test, x_test_id, param_grid=None):
    if param_grid is not None:
        search = RandomizedSearchCV(model, param_grid, n_iter=15, random_state=42)
        search.fit(x_train, y_train)
        best_model = model.__class__(**search.best_params_, random_state=42)
        best_model.fit(x_train, y_train)
        pred = best_model.predict(x_test)
    else:
        model.fit(x_train, y_train)
        pred = model.predict(x_test)
    
    predictions[model_name] = pd.DataFrame({
        "Id": x_test_id,
        "SalePrice": pred
    })

In [56]:
# Random Forest
train_predict_store(RandomForestRegressor(random_state=42), "RandomForest_Default", x_train, y_train, x_test_features, x_test_id)
param_grid_rfr = {'n_estimators': range(5,150,5),
                  'min_samples_split': range(2,100,2),
                  'max_depth': range(1,20,2)}
train_predict_store(RandomForestRegressor(random_state=42), "RandomForest_Hyper", x_train, y_train, x_test_features, x_test_id, param_grid_rfr)

In [57]:
# Decision Tree
train_predict_store(DecisionTreeRegressor(random_state=42), "DecisionTree_Default", x_train, y_train, x_test_features, x_test_id)
param_grid_dtr = {'min_samples_split': range(2,50,2),
                  'max_depth': range(1,20,2)}
train_predict_store(DecisionTreeRegressor(random_state=42), "DecisionTree_Hyper", x_train, y_train, x_test_features, x_test_id, param_grid_dtr)

In [58]:
# MLP Regressor
train_predict_store(MLPRegressor(max_iter=500, random_state=42), "MLP_Default", x_train, y_train, x_test_features, x_test_id)
param_grid_mlp = {'hidden_layer_sizes': [(i,) for i in range(50,200,10)]}
train_predict_store(MLPRegressor(max_iter=500, random_state=42), "MLP_Hyper", x_train, y_train, x_test_features, x_test_id, param_grid_mlp)

c:\Users\knith\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [59]:
# Save all predictions to a single Excel file
output_file = "data/output_house_price.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, df in predictions.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"All model predictions saved to {output_file}")

All model predictions saved to data/output_house_price.xlsx
